In [1]:
# multi class classification of FGL sources using tree categories
import numpy as np
import pandas as pd
from matplotlib import pyplot
from matplotlib.patches import Rectangle
from scipy.stats import binom

from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_curve, roc_auc_score

import igraph
from igraph import Graph, EdgeSeq, plot

import os
import sys
if '../lib/' not in sys.path:
    sys.path.append('../lib/')
import dio3
import multi_fgl_utils as util
import plot_util as putil

In [2]:
igraph.__version__

'0.9.11'

In [ ]:
if 1:
    import importlib
    importlib.reload(util);


In [ ]:
def mlog_gauss_2d(mean, covar):
    H = np.linalg.inv(covar)
    #print(covar, H)
    sign, logdet = np.linalg.slogdet(H)
    logf = 0.5 * logdet - np.log(2. * np.pi)
    def f(x):
        arg = np.dot((x - mean), H)
        arg = np.dot(arg, (x - mean).T)
        #return np.exp(-0.5 * arg)
        #return -0.5 * np.diag(arg)
        return np.diag(arg), logf
    return f

In [ ]:
# load the cat tree
folder = '../data_out/'
tree_alg = 'RF' # 'GMM', 'RF', '4class', 'RF_aakash'
mode = 'all_assoc' # 'all_assoc', 'no_bcu_spp'
if mode == 'all_assoc':
    mode_str = ''
else:
    mode_str = '_%s' % mode


get_models = False
if tree_alg == 'GMM':
    nmin = 100 # 50, 100
    tree_cat_fn = folder + 'tree_cat_nmin%i%s.yaml' % (nmin, mode_str)
    if nmin == 100:
        get_models = True
elif tree_alg == 'RF':
    if mode == 'no_bcu_spp':
        nmin = 60
    else:
        nmin = 100
    tree_cat_fn = folder + 'tree_cat_nmin%i_RF%s.yaml' % (nmin, mode_str)
elif tree_alg == 'RF_aakash':
    nmin = None
    tree_cat_fn = folder + 'tree_cat_rf_15_50.yaml'
elif tree_alg == '4class':
    nmin = None
    tree_cat_fn = folder + 'tree_cat_4class.yaml'

tree_cat_dict = dio3.loaddict(tree_cat_fn)

Eind = 1000
xdf_fn = folder + 'xdf_fgl_Eind%iMeV.csv' % Eind
class_name = 'CLASS1'
#xdf_fn = folder + 'xdf.csv'
xdf = pd.read_csv(xdf_fn, index_col=0)
features = [name for name in xdf.columns if name != class_name]
ns = len(xdf)

nsrc_dict = util.get_nsrc_dict(xdf, class_column=class_name, classes='associated')
assoc_classes = list(nsrc_dict.keys())

#assoc_classes = []
#for cls in tree_cat_dict.values():
#    assoc_classes.extend(cls)
#nsrc_dict = {cl:np.sum(xdf['class'] == cl) for cl in assoc_classes}

if 0:
    alg = 'RF'
    #pars_dict = {'RF':15, 'BDT':9, 'NN':15, "LR":1000}
    pars_dict = {'RF':{'max_depth': 15, 'n_estimators': 50,},
                 'NN':{'activation': 'tanh', 'hidden_layer_sizes': (20, 10,),},
                 'BDT':{'max_depth': 15},
                 'LR':{'max_iter':1000},
                }
    test_frac = .3
    max_depth = 4

tree_plot_pars = {'left':0.08, 'right':0.97, 'bottom':0.12, 'top':0.95, 
                  'font.size':11, 'fig_size':(9, 6)}
if nmin == 15:
    tree_plot_pars['font.size'] = 12
    
domain_plot_pars = {}

markers = ['s', 'o', 'v', '^', 'd', '<', '>',]
colors = pyplot.rcParams['axes.prop_cycle'].by_key()['color']

save_plots = 0
copy2tex = 0
alg_descr = '%s' % tree_alg
if tree_alg == 'GMM':
    alg_descr += '_nmin%s' % (nmin)
alg_descr += mode_str

purpose = 'multi-paper' # 'DFG', 'multi-paper'
if purpose == 'multi-paper':
    plots_folder = '../plots/mclass/viz_split_%s/' % alg_descr
elif purpose == 'DFG':
    plots_folder = '../plots/dfg/%s/' % alg_descr


print(plots_folder)
if not os.path.isdir(plots_folder):
    os.mkdir(plots_folder)

paper_folder = '/Users/Dmitry/Work/Fermi/fermi_multiclass_papers/multi_class_paper1\
/multi_class_paper1_git/'

In [ ]:
if tree_alg == 'GMM' and nmin == 100:
    baseline = True
else:
    baseline = False
print(baseline)

In [ ]:
tree_cat_dict

In [ ]:
fname_dict = {'lp_beta':'LP_beta', 
              'log_uef100':r'log10(Unc_Energy_Flux100 [$\rm erg\ cm^{−2} s^{−1}$])',
              'lp_signif_curv':'LP_SigCurv',
              'var_index':'Variability_Index',
             }
if 0:
    fdict['sin_lat'] = np.sin(np.deg2rad(ctab['GLAT']))
    fdict['cos_lon'] = np.cos(np.deg2rad(ctab['GLON']))
    fdict['sin_lon'] = np.sin(np.deg2rad(ctab['GLON']))
    fdict['log_ef100'] = np.log10(ctab['Energy_Flux100'])
    fdict['log_signif'] = np.log10(ctab['Signif_Avg'])
    Eind = 500 # MeV
    fdict['lp_index500'] = util.logpar_index_cat(Eind, ctab)

In [ ]:
split_node = '0'

In [ ]:
if tree_alg != 'GMM' or (not get_models):
    tree_cat_dict = dio3.loaddict(tree_cat_fn)
    mdl_dict = {}
else:
    cat_dict = {cl:"0" for cl in assoc_classes}
    #cat_dict = {cl:[0] for cl in assoc_classes}
    cls_in = assoc_classes
    mdl_dict = {}
    max_depth = 4
    util.get_tree_cats(cat_dict, cls_in, xdf, features=features, nmin=nmin, 
                       max_depth=max_depth, 
                       nsrc_dict=nsrc_dict, mdl_dict=mdl_dict)

    tree_cat_dict1 = util.cat_dict2tree_cat(cat_dict)
    #dio3.savedict(tree_cat_dict, tree_cat_fn)
    
if util.exist_internal(tree_cat_dict):
    util.remove_internal(tree_cat_dict)

cat_tree = list(tree_cat_dict.keys())
#print(mdl_dict.keys())
#print(tree_cat_dict)
#print(cat_tree)

In [ ]:
#tree_cat = util.cat_dict2tree_cat(cat_dict)
#tree_cat = tree_cat_dict
print(len(tree_cat_dict))
for node, cls in tree_cat_dict.items():
    nsrc = np.sum([nsrc_dict[cl] for cl in cls])
    print(node, cls, nsrc)

In [ ]:
# print groups of classes after the split
child_nodes = util.get_children([split_node], cat_tree)
#print(child_nodes)
child_groups = util.sub_tree2tree_cat_dict(child_nodes, cat_tree, tree_cat_dict)
#print(child_groups)
print('Division 1:')
for key, cls in child_groups.items():
    nsrc = np.sum([nsrc_dict[cl] for cl in cls])
    #print(key)
    st = 'Group %s: ' % key[1:]
    for cl in cls:
        st += '%s, ' % cl
    st = st[:-2] + ' (%i sources);' % nsrc
    print(st)
    
child_nodes = util.get_children(child_nodes, cat_tree)
#print(child_nodes)
child_groups = util.sub_tree2tree_cat_dict(child_nodes, cat_tree, tree_cat_dict)
#print(child_groups)
print('\nDivision 2:')
for key, cls in child_groups.items():
    nsrc = np.sum([nsrc_dict[cl] for cl in cls])
    #print(key)
    st = '%s: ' % key
    for cl in cls:
        st += '%s, ' % cl
    st = st[:-2] + ' (%i sources);' % nsrc
    st += r' \\'
    print(st)

child_nodes = util.get_children(child_nodes, cat_tree, mode='add_0')
#print(child_nodes)
child_groups = util.sub_tree2tree_cat_dict(child_nodes, cat_tree, tree_cat_dict, 
                                           include_parents=True)
print(child_groups)
print('\nDivision 3:')
for key, cls in child_groups.items():
    nsrc = np.sum([nsrc_dict[cl] for cl in cls])
    #print(key)
    st = '%s: ' % key
    for cl in cls:
        st += '%s, ' % cl
    st = st[:-2] + ' (%i sources);' % nsrc
    st += r' \\'
    print(st)

print('\nFinal tree:')
for key, cls in tree_cat_dict.items():
    nsrc = np.sum([nsrc_dict[cl] for cl in cls])
    #print(key)
    st = '%s: ' % key
    for cl in cls:
        st += '%s, ' % cl
    st = st[:-2] + ' (%i sources);' % nsrc
    st += r' \\'
    print(st)

In [ ]:
mdl_dict

In [ ]:
if mdl_dict:
    mdl = mdl_dict[split_node]
    print(mdl.feature_names_in_)

In [ ]:
if mdl_dict:
    sigmas_joined = np.sqrt(np.diagonal(mdl.covariances_[0]) + np.diagonal(mdl.covariances_[1]))
    diff_signif = np.abs(mdl.means_[1] - mdl.means_[0]) / sigmas_joined
    diff_signif_sort = np.sort(diff_signif)[::-1]
    pfeats = []
    pinds = []
    for i, s in enumerate(diff_signif_sort):
        ind = np.argmin(np.abs(diff_signif - s))
        feature = mdl.feature_names_in_[ind]
        print(feature, s)
        if i < 2:
            pfeats.insert(0, feature)
            pinds.insert(0, ind)
    means = mdl.means_[:, pinds]
    covar = mdl.covariances_[:, pinds, :][:, :, pinds]
    sigmas_joined = sigmas_joined[pinds]
    sigmas = np.sqrt(np.diagonal(covar, axis1=1, axis2=2))
    print(means)
    print(sigmas)

In [ ]:
if mdl_dict:
    subclasses = []
    for node in util.all_leaf_children(cat_tree, split_node):
        subclasses.extend(tree_cat_dict[node])
    subclass_mask = util.get_cl_mask(subclasses, xdf)
    xps = xdf[pfeats[0]][subclass_mask]
    #pfeats[1] = 'LP_index1000MeV'
    yps = xdf[pfeats[1]][subclass_mask]
    print(pfeats)
    print(subclasses)

In [ ]:
if mdl_dict:
    cat_dict = {cl:split_node for cl in subclasses}
    cls_in = subclasses
    mdl_dict = {}
    util.get_tree_cats(cat_dict, cls_in, xdf, features=pfeats, 
                       nmin=nmin, max_depth=1, 
                       nsrc_dict=nsrc_dict, mdl_dict=mdl_dict)
    mdl = mdl_dict[split_node]

In [ ]:
if mdl_dict:
    split_node, cat_dict

In [ ]:
if 0:
    sigmas_joined = np.sqrt(np.diagonal(mdl.covariances_[0]) + np.diagonal(mdl.covariances_[1]))
    diff_signif = np.abs(mdl.means_[1] - mdl.means_[0]) / sigmas_joined
    diff_signif_sort = np.sort(diff_signif)[::-1]
    pfeats = []
    pinds = []
    for i, s in enumerate(diff_signif_sort):
        ind = np.argmin(np.abs(diff_signif - s))
        feature = mdl.feature_names_in_[ind]
        print(feature, s)
        if i < 2:
            pfeats.insert(0, feature)
            pinds.insert(0, ind)
    means = mdl.means_[:, pinds]
    covar = mdl.covariances_[:, pinds, :][:, :, pinds]
    sigmas_joined = sigmas_joined[pinds]
    sigmas = np.sqrt(np.diagonal(covar, axis1=1, axis2=2))
    print(pfeats)
    print(means)
    print(sigmas)

In [ ]:
if 1 and mdl_dict:
    means = mdl.means_
    covar = mdl.covariances_
    sigmas = np.sqrt(np.diagonal(covar, axis1=1, axis2=2))
    sigmas_joined = np.sqrt(np.sum(sigmas**2, axis=0))
    print(means)
    print(sigmas_joined)

In [ ]:
if mdl_dict:
    center = np.mean(means, axis=0)
    nn = 4
    minv = center - nn * sigmas_joined
    maxv = center + (nn + 1) * sigmas_joined
    nx = 100
    ny = 100
    x = np.linspace(minv[0], maxv[0], nx)
    y = np.linspace(minv[1], maxv[1], ny)

    X, Y = np.meshgrid(x, y)
    XY = np.array([X.flatten(), Y.flatten()]).T
    XY_df = pd.DataFrame(data=XY, columns=pfeats)
    #print(center, sigmas)

In [ ]:
if mdl_dict:
    g0 = mlog_gauss_2d(means[0], covar[0])
    g0v, logf0 = g0(XY)
    g0v = g0v.reshape((ny, nx))
    g1 = mlog_gauss_2d(means[1], covar[1])
    g1v, logf1 = g1(XY)
    g1v = g1v.reshape((ny, nx))


In [ ]:
if mdl_dict:
    probs = mdl.predict_proba(XY_df)
    pprob = probs.T[1].reshape(X.shape)

In [ ]:
#save_plots = 1
putil.setup_figure_pars(pars_dict=domain_plot_pars)

if mdl_dict:
    all_black_point = 0

    #plot_classes = ['bll', 'fsrq', 'psr', 'msp', 'pwn', 'gal']
    if split_node == '0':
        plot_classes = ['bll', 'pwn', 'msp', 'glc', 'snr', 'sfr', 'sbg']
    elif split_node == '01':
        plot_classes = ['spp', 'fsrq', 'msp', 'snr', 'pwn',]
    elif split_node == '00':
        plot_classes = ['bll', 'sbg', 'bcu', 'ssrq', ]

    cl_colors = {}
    cl_colors[0] = [[0., t, t/2.] for t in np.linspace(0.8, 0.3, len(plot_classes))]
    cl_colors[1] = [[0.9, t, 0.] for t in np.linspace(0.9, 0.2, len(plot_classes))]

    levels = np.arange(1, 5.1, 1)**2
    lw = 1.5
    #fig = pyplot.figure()
    fig, ax = pyplot.subplots(figsize=(9, 6))
    #pyplot.figure(figsize=(9, 4.5))
    ccmap0 = 'Blues'# 'winter_r'
    ccmap1 = 'autumn'
    #pyplot.imshow(-0.5 * (g0v + g1v), origin='lower', 
                    #extent=(minv[0], maxv[0], minv[1], maxv[1]))

    if 1:
        im = ax.imshow(pprob, origin='lower', 
                      #extent=(minv[0], maxv[0], minv[1], maxv[1]), 
                      extent=(x[0], x[-1], y[0], y[-1]),
                      cmap='GnBu_r', alpha=0.9, vmin=-0.0001)
        aspect = (x[-1] - x[0])/(y[-1] - y[0]) * 0.7
        ax.set_aspect(aspect)
        fig.colorbar(im)
    xlabel = fname_dict.get(pfeats[0], pfeats[0])
    ylabel = fname_dict.get(pfeats[1], pfeats[1])
    pyplot.xlabel(xlabel)
    pyplot.ylabel(ylabel)

    if all_black_point:
        vmask = (x[0] < xps) & (x[-1] > xps)
        vmask &= (y[0] < yps) & (y[-1] > yps)

        pyplot.scatter(xps[vmask], yps[vmask], marker='x', color='black', s=2)
        pyplot.contour(X, Y, g0v, levels=levels, linewidths=lw, linestyles='--', 
                       cmap=ccmap0, alpha=1)
        pyplot.contour(X, Y, g1v, levels=levels, linewidths=lw, linestyles='-', 
                       cmap=ccmap1, alpha=1)
        #pyplot.colorbar()
    else:
        pyplot.contour(X, Y, g0v, levels=levels, linewidths=lw, linestyles='--', 
                       cmap=ccmap0, alpha=1)
        pyplot.contour(X, Y, g1v, levels=levels, linewidths=lw, linestyles='-', 
                       cmap=ccmap1, alpha=1)
        for i, cls in enumerate(plot_classes):
            cmask = (xdf[class_name] == cls)
            xpsc = xdf[cmask][pfeats[0]]
            ypsc = xdf[cmask][pfeats[1]]
            vmask = (x[0] < xpsc) & (x[-1] > xpsc)
            vmask &= (y[0] < ypsc) & (y[-1] > ypsc)
            xpsc = xpsc[vmask]
            ypsc = ypsc[vmask]
            color = cl_colors[int(cat_dict[cls][-1])][i]
            print(cls, int(cat_dict[cls][-1]))
            #if i == 0:
            #    color = 'blue'
            #else:
            #    color = colors[i%len(colors)]

            if np.sum(cmask) < 10:
                size = 50
                #print(cls, np.sum(cmask))
                #color = 'black'
            elif np.sum(cmask) < 50:
                size = 30
                #print(cls, np.sum(cmask))
            else:
                size = 20
            pyplot.scatter(xpsc, ypsc, marker=markers[i%len(markers)], s=size, 
                           label=cls, color=color)
        pyplot.legend(loc='upper right')

    if save_plots:
        if all_black_point:
            fn = '%s/split_node%s_all_black_%s.pdf' % (plots_folder, split_node, alg_descr)
        else:
            fn = '%s/split_node%s_classes_%s.pdf' % (plots_folder, split_node, alg_descr)
        print('save figure to file: %s' % (fn))
        pyplot.savefig(fn)
        fn_png = fn.replace('.pdf', '.png')
        pyplot.savefig(fn_png)
        if copy2tex:
            plots_folder_paper = paper_folder + 'plots/viz_split/'
            fn_paper = fn.replace(plots_folder, plots_folder_paper)
            cmd = 'cp %s %s' % (fn, fn_paper)
            print(cmd)
            os.system(cmd)


In [ ]:
if mdl_dict:
    # print probabilities
    pred = {}
    for i, cls in enumerate(plot_classes):
        cmask = (xdf[class_name] == cls)
        X_ps = xdf[cmask][pfeats]
        probs = mdl.predict_proba(X_ps)
        pp = np.mean(probs, axis=0)
        pred[cls] = probs.T[1]
        print(r'%s & %s & %.3f & %i \\' % \
               (cls, cat_dict[cls][-1], pp[1], sum(cmask)))

    print()
    cts, thres = util.get_cats(pred, thres=0.5, out_thres=True)
    print(cts)
    print('%.3f' % thres)

In [ ]:
if mdl_dict:
    horiz_trees = util.get_horiz_trees(cat_tree)
    subtree = horiz_trees[len(split_node) - 1]
    print(subtree)
    cut_subtree = util.sub_tree2tree_cat_dict(subtree, cat_tree, tree_cat_dict)
    print('2D groups')
    for key, cls in cut_subtree.items():
        nsrc = np.sum([nsrc_dict[cl] for cl in cls])
        #print(key)
        st = 'Group %s: ' % key[1:]
        for cl in cls:
            st += '%s, ' % cl
        st = st[:-2] + ' (%i sources);' % nsrc
        print(st)

In [ ]:
#horiz_trees

In [ ]:
g = Graph(directed=True)
root = '0'
g.add_vertices(root)
if util.is_internal(root, tree=cat_tree):
    internal_nodes = [root]
    
while len(internal_nodes) > 0:
    node = internal_nodes.pop(0)
    if util.is_internal(node, tree=cat_tree):
        child_nodes = util.get_children([node], cat_tree)
        #adict = {'label': child_nodes}
        #g.add_vertices(child_nodes, attributes=adict)
        g.add_vertices(child_nodes)
        for cnode in child_nodes:
            g.add_edge(node, cnode)
            if util.is_internal(cnode, tree=cat_tree):
                internal_nodes.append(cnode)

In [ ]:
#layout = g.layout("tree")
layout = g.layout("rt")
nv = len(g.vs["name"])
max_depth_tree = util.max_node_length(cat_tree) - 1

visual_style = {}
visual_style["vertex_label"] = g.vs["name"]
visual_style["vertex_color"] = []
if 1:
    udict = {}
    for i, lb in enumerate(visual_style["vertex_label"]):
        if lb in tree_cat_dict:
            lb_new = '%s' % lb
            for k, cl in enumerate(tree_cat_dict[lb]):
                if k % 2 == 0: # and lb != '01111'or k == 0:
                    lb_new += '\n'
                lb_new += '%s ' % cl
            udict[i] = lb_new[:-1]
            visual_style["vertex_color"].append(colors[1])
        else:
            visual_style["vertex_color"].append(colors[0])
    for i, lb in udict.items():
        visual_style["vertex_label"][i] = lb
    visual_style["vertex_size"] = 10

#plot(g, layout=layout)

In [ ]:
#pyplot.rcParams['font.size'] = 11
#pyplot.rcParams['figure.subplot.bottom'] = 0.05
#pyplot.rcParams['figure.subplot.top'] = 0.95
putil.setup_figure_pars(pars_dict=tree_plot_pars)

fig, ax = pyplot.subplots(figsize=(9, 6))
plot(g, layout=layout, target=ax, **visual_style)
# move texts around
for t in ax.texts:
    if t.get_text().find('\n') > -1:
        t.set_verticalalignment('top')
        t.set_horizontalalignment('left')
        x_new = t._x - 0.1
        t.set_x(x_new)
        y_new = t._y + 0.2
        t.set_y(y_new)
    else:
        t.set_verticalalignment('center')
        t.set_horizontalalignment('left')
        x_new = t._x + 0.1
        t.set_x(x_new)
        y_new = t._y + 0.0
        t.set_y(y_new)
        
pyplot.ylim(max_depth_tree + 1.2, -0.5)
pyplot.xlim(-3, 5.2)
ax.set_xticks([]);
ax.set_yticks(range(0, max_depth_tree + 1));
pyplot.ylabel('Depth')
if save_plots:
    fn = '%s/split_tree_nmin%i.pdf' % (plots_folder, nmin)
    print('save figure to file: %s' % (fn))
    pyplot.savefig(fn)
    fn_png = fn.replace('.pdf', '.png')
    pyplot.savefig(fn_png)
    if copy2tex and 0:
        plots_folder_paper = paper_folder + 'plots/viz_split/'
        fn_paper = fn.replace(plots_folder, plots_folder_paper)
        cmd = 'cp %s %s' % (fn, fn_paper)
        print(cmd)
        os.system(cmd)


In [ ]:
# plot horiz trees
g = Graph(directed=True)
root = '0'
g.add_vertices(root)
horiz_nodes_next = [root]

for i in range(max_depth_tree):
    horiz_nodes = horiz_nodes_next.copy()
    horiz_nodes_next = []
    for node in horiz_nodes:
        child_nodes = util.get_children([node], cat_tree, mode='add_0')
        for cnode in child_nodes:
            g.add_vertices(cnode)
            g.add_edge(node, cnode)
            horiz_nodes_next.append(cnode)


In [ ]:
# put 'nlsy1' at the end of the list for nice plotting
for node, cls in tree_cat_dict.items():
    if 'nlsy1' in cls:
        cls.remove('nlsy1')
        cls.append('nlsy1')

add_nsrc = 1
#layout = g.layout("tree")
layout = g.layout("rt")

visual_style = {}
visual_style["vertex_label"] = []
visual_style["vertex_color"] = []
for i, node in enumerate(g.vs["name"]):
    child_nodes = util.get_children([node], cat_tree, mode='add_0')
    if len(child_nodes) == 2:
        visual_style["vertex_color"].append(colors[0])
        label = node
        if tree_alg == '4class':
            if node == '0':
                label += ' - "all sources"'
            elif node == '00':
                label += ' - extragalactic'
            elif node == '01':
                label += ' - Galactic'
        visual_style["vertex_label"].append(label)
    else:
        if node in tree_cat_dict:
            visual_style["vertex_color"].append(colors[1])
        else:
            visual_style["vertex_color"].append(colors[2])
        if len(node) != max_depth_tree + 1:
            visual_style["vertex_label"].append(node)
        else:
            label = '%s' % node
            while node not in tree_cat_dict and len(node) > 0:
                print(node)
                node = node[:-1]
            nsrc = 0
            for k, cl in enumerate(tree_cat_dict[node]):
                nsrc += nsrc_dict[cl]
                if len(tree_cat_dict[node]) < 7:
                    div = 2
                else:
                    div = 3
                if k % div == 0:
                    label += '\n'
                label += '%s, ' % cl
            label = label[:-2]
            if add_nsrc:
                label += '\n' + r'$n_{s} = %i$' % nsrc
            visual_style["vertex_label"].append(label)

visual_style["vertex_size"] = 10
if tree_alg == '4class':
    visual_style["vertex_label_size"] = 14
elif nmin == 15:
    visual_style["vertex_label_size"] = 12
else:
    visual_style["vertex_label_size"] = 13
visual_style['layout'] = layout

In [ ]:
#copy2tex = 1
baseline

In [ ]:
if purpose == 'DFG':
    select_level = 2 # None, 3
else:
    if baseline:
        select_level = 2
    else:
        select_level = None
    
if nmin == 15:
    if max_depth_tree == 5:
        xmin = -4
        xmax = 7.5
    elif max_depth_tree == 4:
        xmin = -3.6
        xmax = 5.5
    else:
        xmin = -3.5
        xmax = 4
        
elif nmin == 50:
    xmin = -2.5
    xmax = 5
elif nmin == 100:
    if tree_alg == 'RF':
        xmin = -4.5
        xmax = 2.
    else:
        xmin = -2.5
        xmax = 4.2
elif tree_alg == '4class':
    xmin = -2.5
    xmax = 2.5
elif tree_alg == 'RF_aakash':
    xmin = -5
    xmax = 2
elif mode == 'no_bcu_spp':
    if tree_alg == 'RF':
        xmin = -2.5
        xmax = 2.5

ymin = -0.5
if tree_alg == '4class':
    ymax = 3
elif add_nsrc:
    ymax = max_depth_tree + 1.6
    if tree_alg == 'RF':
        if mode == 'no_bcu_spp':
            ymax += 0.0
        else:
            ymax += 0.5
    elif tree_alg == 'RF_aakash':
        ymax += 1.
else:
    ymax = max_depth_tree + 1.2


putil.setup_figure_pars(pars_dict=tree_plot_pars)
#pyplot.rcParams['font.size'] = 12
#pyplot.rcParams['figure.subplot.bottom'] = 0.12
#pyplot.rcParams['figure.subplot.top'] = 0.95
#fig, ax = pyplot.subplots(figsize=(9, 6))
fig, ax = pyplot.subplots()

plot(g, target=ax, **visual_style)

if select_level is not None:
    xy = (xmin+0.3, select_level - 0.5)
    width = xmax - xmin - 0.6
    height = 1.
    ax.add_patch(Rectangle(xy, width, height, angle=0.0, fill=False, ls='--'))

for t in ax.texts:
    if t.get_text().find('\n') > -1:
        t.set_verticalalignment('top')
        if tree_alg == '4class':
            t.set_horizontalalignment('left')
            x_new = t._x - 0.05
            y_new = t._y + 0.1
        else:
            t.set_horizontalalignment('left')
            x_new = t._x - 0.2
            y_new = t._y + 0.2
        t.set_x(x_new)
        t.set_y(y_new)
    else:
        t.set_verticalalignment('center')
        t.set_horizontalalignment('left')
        x_new = t._x + 0.1
        t.set_x(x_new)
        y_new = t._y + 0.0
        t.set_y(y_new)

        

pyplot.ylim(ymax, ymin)
pyplot.xlim(xmin, xmax)
if nmin == 15:
    if max_depth_tree == 5:
        ax.set_xticks(np.arange(-3.5, 6.6, 1));
    if max_depth_tree == 4:
        ax.set_xticks(np.arange(-3.2, -3.2 + len(tree_cat_dict), 1));
    elif max_depth_tree == 3:
        ax.set_xticks(np.arange(xmin + 0.5, xmax - 1 + 0.1, 1));
    ax.set_xticklabels(np.arange(1, len(tree_cat_dict) + 1, 1));
elif nmin == 50:
    ax.set_xticks(np.arange(-1.9, 4.3, 1));
    ax.set_xticklabels(np.arange(1, len(tree_cat_dict) + 1, 1));
elif nmin == 100:
    if tree_alg == 'RF':
        ax.set_xticks(np.arange(xmin + 0.8, xmax, 1));
    else:
        ax.set_xticks(np.arange(-1.8, 3.3, 1));
    ax.set_xticklabels(np.arange(1, len(tree_cat_dict) + 1, 1));
elif tree_alg == '4class':
    ax.set_xticks([]);
    #ax.set_xticklabels(np.arange(0, len(tree_cat_dict), 1));
elif mode == 'no_bcu_spp':
    if tree_alg == 'RF':
        ax.set_xticks(np.arange(-2.1, 2.1, 1));
        ax.set_xticklabels(np.arange(1, len(tree_cat_dict) + 1, 1));

        
ax.set_yticks(range(0, max_depth_tree + 1));
if tree_alg != '4class':
    pyplot.xlabel('Classification groups')
pyplot.ylabel('Depth')

if save_plots:
    if select_level is not None:
        fn = '%s/split_tree_full_select_level%i_nmin%i.pdf' % \
                (plots_folder, select_level, nmin)
    else:
        fn = '%s/split_tree_full_%s.pdf' % (plots_folder, alg_descr)
    print('save figure to file: %s' % (fn))
    pyplot.savefig(fn)
    fn_png = fn.replace('.pdf', '.png')
    pyplot.savefig(fn_png)
    if copy2tex:
        plots_folder_paper = paper_folder + 'plots/viz_split/'
        fn_paper = fn.replace(plots_folder, plots_folder_paper)
        cmd = 'cp %s %s' % (fn, fn_paper)
        print(cmd)
        os.system(cmd)


In [ ]:
max_depth_tree, xmin, nmin, len(tree_cat_dict)

In [ ]:
len(tree_cat_dict)

In [ ]:
print(dir(ax))